# Plant Disease AI — Exploratory Data Analysis

This notebook covers:
1. Class distribution analysis
2. Image quality statistics
3. Sample visualization per disease
4. Feature space visualization (t-SNE on embeddings)
5. Color distribution analysis (healthy vs diseased)
6. Data augmentation preview

In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import random
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
import torch
import torchvision.models as models
from torchvision.datasets import ImageFolder
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../data/processed/plantvillage')
print(f'Data directory: {DATA_DIR}')
print(f'Exists: {DATA_DIR.exists()}')

## 1. Class Distribution

In [ ]:
class_counts = {}
for class_dir in sorted(DATA_DIR.iterdir()):
    if class_dir.is_dir():
        count = len(list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.png')))
        class_counts[class_dir.name] = count

df = pd.DataFrame(list(class_counts.items()), columns=['class', 'count']).sort_values('count')
df['crop'] = df['class'].apply(lambda x: x.split('___')[0])
df['disease'] = df['class'].apply(lambda x: x.split('___')[1].replace('_', ' ').title())
df['is_healthy'] = df['class'].str.endswith('healthy')

print(f'Total classes   : {len(df)}')
print(f'Total images    : {df["count"].sum():,}')
print(f'Healthy classes : {df["is_healthy"].sum()}')
print(f'Disease classes : {(~df["is_healthy"]).sum()}')
print(f'\nImbalance ratio (max/min): {df["count"].max()/df["count"].min():.1f}x')
print(f'Mean images/class: {df["count"].mean():.0f}')
print(f'Median images/class: {df["count"].median():.0f}')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Horizontal bar chart — class counts
colors = ['#2ecc71' if h else '#e74c3c' for h in df['is_healthy']]
axes[0].barh(range(len(df)), df['count'], color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)
axes[0].set_yticks(range(len(df)))
axes[0].set_yticklabels([f"{r['crop'].title()}: {r['disease']}" for _, r in df.iterrows()], fontsize=7)
axes[0].set_xlabel('Number of Images')
axes[0].set_title('Class Distribution (Green = Healthy, Red = Disease)', fontweight='bold')
axes[0].axvline(df['count'].mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Mean ({df["count"].mean():.0f})')
axes[0].legend()

# Crop-level pie chart
crop_totals = df.groupby('crop')['count'].sum().sort_values(ascending=False)
axes[1].pie(crop_totals.values, labels=[c.title() for c in crop_totals.index],
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Set3', len(crop_totals)))
axes[1].set_title('Images by Crop Type', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Sample Images per Disease

In [ ]:
SHOW_CLASSES = [
    'tomato___early_blight', 'tomato___late_blight', 'tomato___healthy',
    'potato___early_blight', 'potato___late_blight', 'potato___healthy',
    'corn___northern_leaf_blight', 'corn___common_rust', 'corn___healthy',
]

fig, axes = plt.subplots(len(SHOW_CLASSES), 4, figsize=(14, 3 * len(SHOW_CLASSES)))

for row_idx, cls in enumerate(SHOW_CLASSES):
    cls_dir = DATA_DIR / cls
    if not cls_dir.exists():
        print(f'Missing: {cls}')
        continue
    images = list(cls_dir.glob('*.jpg'))[:4]
    random.shuffle(images)
    for col_idx, img_path in enumerate(images[:4]):
        ax = axes[row_idx][col_idx]
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        ax.axis('off')
        if col_idx == 0:
            label = cls.replace('___', '\n').replace('_', ' ').title()
            ax.set_ylabel(label, fontsize=9, rotation=0, labelpad=80, va='center')

plt.suptitle('Sample Images by Disease Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Image Quality Statistics

In [ ]:
SAMPLE_SIZE = 500  # images to analyze (full dataset would take too long in notebook)

quality_data = []
all_images = list(DATA_DIR.glob('**/*.jpg'))
sampled = random.sample(all_images, min(SAMPLE_SIZE, len(all_images)))

for img_path in tqdm(sampled, desc='Analyzing quality'):
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        continue
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    mean_b = gray.mean()
    h, w = bgr.shape[:2]
    cls = img_path.parent.name
    quality_data.append({
        'class': cls,
        'crop': cls.split('___')[0],
        'sharpness': lap_var,
        'brightness': mean_b,
        'width': w,
        'height': h,
        'is_healthy': cls.endswith('healthy'),
    })

qdf = pd.DataFrame(quality_data)
print(qdf[['sharpness', 'brightness', 'width', 'height']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(qdf['sharpness'].clip(upper=2000), bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Laplacian Variance (Sharpness)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sharpness Distribution')
axes[0].axvline(qdf['sharpness'].median(), color='red', linestyle='--', label=f'Median: {qdf["sharpness"].median():.0f}')
axes[0].legend()

axes[1].hist(qdf['brightness'], bins=50, color='goldenrod', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Mean Brightness (0-255)')
axes[1].set_title('Brightness Distribution')

axes[2].scatter(qdf['sharpness'].clip(upper=3000), qdf['brightness'],
                c=['#2ecc71' if h else '#e74c3c' for h in qdf['is_healthy']],
                alpha=0.4, s=15)
axes[2].set_xlabel('Sharpness')
axes[2].set_ylabel('Brightness')
axes[2].set_title('Sharpness vs Brightness\n(Green=Healthy, Red=Disease)')

plt.tight_layout()
plt.savefig('../reports/image_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Augmentation Preview

In [ ]:
from backend.app.ml.preprocessing.image_preprocessor import ImagePreprocessor
import torchvision.transforms.functional as TF

preprocessor = ImagePreprocessor(target_size=(380, 380))

# Pick one sample image
sample_class = 'tomato___early_blight'
sample_imgs = list((DATA_DIR / sample_class).glob('*.jpg'))[:1]
if sample_imgs:
    orig = Image.open(sample_imgs[0]).convert('RGB')

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    axes = axes.flatten()

    axes[0].imshow(orig.resize((224, 224)))
    axes[0].set_title('Original', fontweight='bold')
    axes[0].axis('off')

    for i in range(1, 12):
        augmented = preprocessor.train_transform(orig)
        # Denormalize for display
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        denorm = (augmented * std + mean).clamp(0, 1)
        pil_aug = TF.to_pil_image(denorm)
        axes[i].imshow(pil_aug)
        axes[i].set_title(f'Aug #{i}', fontsize=9)
        axes[i].axis('off')

    plt.suptitle(f'Training Augmentations — {sample_class}', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/augmentation_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print(f'No images found in {DATA_DIR / sample_class}')

## 5. Color Analysis — Healthy vs Diseased

In [ ]:
PAIRS = [
    ('tomato___healthy', 'tomato___early_blight'),
    ('potato___healthy', 'potato___late_blight'),
    ('corn___healthy', 'corn___northern_leaf_blight'),
]

fig, axes = plt.subplots(len(PAIRS), 3, figsize=(15, 5 * len(PAIRS)))

for row, (healthy_cls, disease_cls) in enumerate(PAIRS):
    for col, (cls, label, color) in enumerate([
        (healthy_cls, 'Healthy', '#2ecc71'),
        (disease_cls, 'Diseased', '#e74c3c'),
        (None, 'HSV-Hue Comparison', None),
    ]):
        ax = axes[row][col]
        if cls is not None:
            imgs = list((DATA_DIR / cls).glob('*.jpg'))[:30]
            hues = []
            for p in imgs:
                bgr = cv2.imread(str(p))
                if bgr is not None:
                    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
                    hues.extend(hsv[:, :, 0].flatten().tolist())
            ax.hist(hues, bins=90, range=(0, 180), color=color, alpha=0.7, density=True)
            ax.set_xlabel('Hue (0-180 in OpenCV)')
            ax.set_ylabel('Density')
            crop = cls.split('___')[0].title()
            ax.set_title(f'{crop} — {label}')
        else:
            # Overlay both on same axes
            for sub_cls, sub_label, sub_color in [
                (healthy_cls, 'Healthy', '#2ecc71'),
                (disease_cls, 'Diseased', '#e74c3c'),
            ]:
                imgs = list((DATA_DIR / sub_cls).glob('*.jpg'))[:30]
                hues = []
                for p in imgs:
                    bgr = cv2.imread(str(p))
                    if bgr is not None:
                        hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
                        hues.extend(hsv[:, :, 0].flatten().tolist())
                ax.hist(hues, bins=90, range=(0, 180), alpha=0.5, density=True, label=sub_label, color=sub_color)
            ax.legend()
            ax.set_xlabel('Hue (0-180)')
            ax.set_title(f'{healthy_cls.split("___")[0].title()} Hue Overlay')

plt.suptitle('Color Distribution: Healthy vs Diseased Leaves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/color_analysis.png', dpi=150, bbox_inches='tight')
plt.show()